# Reported pedestrian injury collisions: initial data inspection

This notebook checks whether the supplied Department for Transport (DfT) collision and casualty files support a one-borough concentration study. It does not select a borough, run hotspot analysis or fit a predictive model.

**Status:** this notebook currently contains input checks, example records, annual header comparison and a column dictionary. Section 1.3 has saved results; later analysis has not been included. Fatal, serious and slight pedestrian casualties are included; each qualifying collision is counted once. Counts describe reported collisions, not pedestrian risk.

## Task 1: Data inspection and study-scope assessment

### 1.1 Working directory and inputs

Open this notebook with the working directory set to the repository’s `notebooks/` folder. The default input directory is `../data`, relative to that working directory. Set `DATA_DIR` or the `COMP1890_DATA_DIR` environment variable to use another input location. Keep the `notebooks/` and `data/` folders together when packaging the project; no particular repository name or absolute location is required.

The required inputs are the eight annual CSVs, the Excel guide and `download-manifest.json`. Use the **Python (London pedestrian collisions)** kernel from the project environment, with dependencies documented in `../requirements.txt`. This notebook reads inputs into memory and displays tables; it does not modify inputs or write derived datasets.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import platform

import pandas as pd
from IPython.display import display

DATA_DIR = Path(os.environ.get("COMP1890_DATA_DIR", "../data")).expanduser()
YEARS = (2022, 2023, 2024, 2025)
GUIDE_NAME = "dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2025.xlsx"

if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        "Input directory not found. Run from the notebook folder and set DATA_DIR "
        "or COMP1890_DATA_DIR to the supplied input directory."
    )
print({"python": platform.python_version(), "pandas": pd.__version__})


{'python': '3.12.3', 'pandas': '3.1.1'}


### 1.2 File inventory and integrity

Check that all required files exist and that their sizes and SHA-256 checksums match the download manifest. This establishes consistency with the preserved download, not the correctness or release status of its records. Stop if verification fails.


In [2]:
manifest_path = DATA_DIR / "download-manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("download-manifest.json is missing from the input directory.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
entries = {entry["file"]: entry for entry in manifest["files"]}
expected_files = [
    f"dft-road-casualty-statistics-{kind}-{year}.csv"
    for year in YEARS for kind in ("collision", "casualty")
] + [GUIDE_NAME]
checks = []
for name in expected_files:
    path = DATA_DIR / name
    if not path.is_file() or name not in entries:
        raise FileNotFoundError(f"Required input or manifest entry missing: {name}")
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    entry = entries[name]
    checks.append({
        "file": name,
        "bytes": path.stat().st_size,
        "size_matches": path.stat().st_size == entry["bytes"],
        "sha256_matches": digest.hexdigest() == entry["sha256"],
    })
file_checks = pd.DataFrame(checks)
display(file_checks)
if not file_checks[["size_matches", "sha256_matches"]].all().all():
    raise ValueError("Input verification failed; resolve differences before continuing.")
print("Source page:", manifest["source_page"])


,file,bytes,size_matches,sha256_matches
0,dft-road-casualty-statistics-collision-2022.csv,20116923,True,True
1,dft-road-casualty-statistics-casualty-2022.csv,11242293,True,True
2,dft-road-casualty-statistics-collision-2023.csv,19790001,True,True
3,dft-road-casualty-statistics-casualty-2023.csv,11012867,True,True
4,dft-road-casualty-statistics-collision-2024.csv,19208488,True,True
5,dft-road-casualty-statistics-casualty-2024.csv,10439101,True,True
6,dft-road-casualty-statistics-collision-2025.csv,19349153,True,True
7,dft-road-casualty-statistics-casualty-2025.csv,10318523,True,True
8,dft-road-casualty-statistics-road-safety-open-...,85112,True,True


Source page: https://www.gov.uk/government/statistical-data-sets/road-safety-open-data


**Findings:** pending execution. Record any missing or changed files after reviewing the output.

### 1.3 Example records and annual column comparison

Show the first five rows of the 2022 collision file and the 2022 casualty file, with all columns visible. Use one example per file type to avoid repeating similar tables. Read only headers for the annual schema comparison: compare collision columns against collision columns, and casualty columns against casualty columns, using 2022 as the baseline.

Report column counts, added or missing names and column order for every year. Equal columns do not imply equal records, field definitions, missingness or reporting completeness; those require the later checks. No original inputs are changed.


In [2]:
PREVIEW_YEAR = 2022
FILE_TYPES = ("collision", "casualty")

with pd.option_context("display.max_columns", None):
    for kind in FILE_TYPES:
        name = f"dft-road-casualty-statistics-{kind}-{PREVIEW_YEAR}.csv"
        print(f"\nFirst five rows: {name}")
        preview = pd.read_csv(
            DATA_DIR / name,
            nrows=5,
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig",
        )
        display(preview)

schema_rows = []
for kind in FILE_TYPES:
    baseline_name = f"dft-road-casualty-statistics-{kind}-{PREVIEW_YEAR}.csv"
    baseline_columns = pd.read_csv(
        DATA_DIR / baseline_name, nrows=0, encoding="utf-8-sig"
    ).columns.tolist()
    for year in YEARS:
        name = f"dft-road-casualty-statistics-{kind}-{year}.csv"
        columns = pd.read_csv(
            DATA_DIR / name, nrows=0, encoding="utf-8-sig"
        ).columns.tolist()
        schema_rows.append({
            "file_type": kind,
            "year": year,
            "column_count": len(columns),
            "same_column_names_as_2022": set(columns) == set(baseline_columns),
            "same_column_order_as_2022": columns == baseline_columns,
            "added_columns": sorted(set(columns) - set(baseline_columns)),
            "missing_columns": sorted(set(baseline_columns) - set(columns)),
        })

schema_comparison = pd.DataFrame(schema_rows)
print("\nAnnual column comparison (2022 baseline for each file type)")
with pd.option_context("display.max_columns", None):
    display(schema_comparison)

for kind in FILE_TYPES:
    comparison = schema_comparison.loc[schema_comparison["file_type"].eq(kind)]
    names_match = comparison["same_column_names_as_2022"].all()
    order_matches = comparison["same_column_order_as_2022"].all()
    print(f"{kind}: all years have the same column names = {bool(names_match)}; "
          f"same order = {bool(order_matches)}; "
          f"column counts = {comparison['column_count'].tolist()}")



First five rows: dft-road-casualty-statistics-collision-2022.csv


,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,local_authority_district,local_authority_ons_district,local_authority_highway,local_authority_highway_current,first_road_class,first_road_number,road_type,speed_limit,junction_detail_historic,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control_historic,pedestrian_crossing_physical_facilities_historic,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
0,2022170M11542,2022,170M11542,449805,520540,-1.231017,54.577627,17,3,3,3,23/09/2022,6,08:20,-1,E06000002,E06000002,E06000002,3,66,3,50,5,0,4,3,172,0,0,0,1,1,1,0,0,0,2,1,2,E01033468,-1,0,0.1228,0.8772
1,2022170S41352,2022,170S41352,442918,516384,-1.338157,54.54091,17,3,3,2,06/09/2022,3,16:05,-1,E06000004,E06000004,E06000004,3,135,6,40,3,13,4,6,0,0,0,0,1,1,2,0,0,0,2,1,2,E01012269,-1,0,0.07984,0.92016
2,2022070018133,2022,070018133,359806,391896,-2.606293,53.422321,7,3,2,1,08/01/2022,7,17:55,-1,E06000007,E06000007,E06000007,1,62,3,70,0,0,-1,0,-1,0,0,0,7,5,2,0,0,0,1,1,1,E01012468,-1,0,0.03815,0.96185
3,2022041207140,2022,041207140,334066,433587,-3.002406,53.794398,4,3,1,1,08/08/2022,2,22:31,-1,E06000009,E06000009,E06000009,3,5230,1,40,1,0,4,6,0,0,0,0,4,1,1,0,0,0,2,1,2,E01012688,3,1,0,1
4,2022041230134,2022,041230134,333517,437160,-3.011512,53.826437,4,3,1,1,13/10/2022,5,16:42,-1,E06000009,E06000009,E06000009,4,5266,6,30,9,19,4,6,0,0,0,0,1,1,1,0,0,0,1,1,2,E01012722,3,1,0,1



First five rows: dft-road-casualty-statistics-casualty-2022.csv


,collision_index,collision_year,collision_ref_no,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,casualty_severity,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_imd_decile,lsoa_of_casualty,enhanced_casualty_severity,casualty_injury_based,casualty_adjusted_severity_serious,casualty_adjusted_severity_slight,casualty_distance_banding,escooter_flag
0,2022010358495,2022,010358495,1,1,3,2,29,6,3,5,1,0,0,0,0,3,E01001045,-1,0,0,1,-1,0
1,2022010360442,2022,010360442,1,1,3,2,8,2,3,1,9,0,0,0,0,2,E01004719,-1,0,0,1,-1,0
2,2022010361982,2022,010361982,1,1,3,2,14,3,3,4,4,0,0,0,0,3,E01002403,-1,0,0,1,-1,0
3,2022010366070,2022,010366070,1,1,3,1,18,4,3,1,1,0,0,0,0,5,E01001908,-1,0,0,1,-1,0
4,2022010366778,2022,010366778,1,1,3,1,36,7,3,4,3,0,0,0,0,7,E01003763,-1,0,0,1,-1,0



Annual column comparison (2022 baseline for each file type)


,file_type,year,column_count,same_column_names_as_2022,same_column_order_as_2022,added_columns,missing_columns
0,collision,2022,44,True,True,[],[]
1,collision,2023,44,True,True,[],[]
2,collision,2024,44,True,True,[],[]
3,collision,2025,44,True,True,[],[]
4,casualty,2022,24,True,True,[],[]
5,casualty,2023,24,True,True,[],[]
6,casualty,2024,24,True,True,[],[]
7,casualty,2025,24,True,True,[],[]


collision: all years have the same column names = True; same order = True; column counts = [44, 44, 44, 44]
casualty: all years have the same column names = True; same order = True; column counts = [24, 24, 24, 24]


**Findings:** the 2022 collision and casualty previews each show five records. Across 2022–2025, all collision files have the same 44 column names in the same order; all casualty files have the same 24 column names in the same order. No columns were added or missing relative to the corresponding 2022 file. The two file types have different schemas, so they are compared separately. Matching headers do not establish record-level quality or stable coding definitions.

**Execution note:** this preview and header comparison was rerun with the setup cell in a separate kernel session. Other saved outputs were preserved; the full notebook was not rerun in this change.

### 1.4 Column descriptions

This dictionary describes every column in the supplied collision and casualty files, in CSV order. It is based on the official [DfT Excel data guide](../data/dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2025.xlsx), sheet `2024_code_list`, inspected during preparation. Download links and provenance are documented in [data.md](../data/data.md).

Numeric-looking values can be identifiers, category codes, ordinal categories, counts or coordinates. Category codes must be interpreted using their labels; they are not continuous measurements. Examples below are not complete code lists. Missing, unknown and not-applicable values are field-specific; neither `0` nor `-1` should be replaced globally.

The annual header check in section 1.3 found matching columns within each file type for 2022–2025. Matching headers do not establish identical definitions or historical availability. Several fields were introduced or converted in later releases. Fields not explained by the supplied code-list sheet are marked explicitly; consult the linked [DfT source and guidance page](https://www.gov.uk/government/statistical-data-sets/road-safety-open-data) before using them.

**Collision table — 44 columns**

| Column | Description | Data kind and interpretation |
| --- | --- | --- |
| `collision_index` | Unique collision identifier combining the year and police reference; joins collision and casualty records. | Identifier; preserve as text. |
| `collision_year` | Year assigned to the collision record. | Year; compare against the date and source-file year. |
| `collision_ref_no` | Police reference within a year; not globally unique across years. | Identifier; use collision_index for cross-year links. |
| `location_easting_osgr` | Ordnance Survey grid easting of the collision location. | Spatial coordinate; the guide states null when unknown. Confirm the coordinate reference system before mapping. |
| `location_northing_osgr` | Ordnance Survey grid northing of the collision location. | Spatial coordinate; the guide states null when unknown. Confirm the coordinate reference system before mapping. |
| `longitude` | Longitude of the collision location. | Spatial coordinate; null when unknown. Negative values can be valid. |
| `latitude` | Latitude of the collision location. | Spatial coordinate; null when unknown. |
| `police_force` | Police force associated with the collision record. | Categorical code; for example, 1 = Metropolitan Police. Use the guide for other forces and historical changes. |
| `collision_severity` | Recorded severity category of the collision. | Ordinal category: 1 = Fatal, 2 = Serious, 3 = Slight. Distinct from individual casualty severity. |
| `number_of_vehicles` | Number of vehicles recorded for the collision. | Count. |
| `number_of_casualties` | Number of casualties recorded for the collision, across road-user types. | Count; not specifically the number of pedestrians. |
| `date` | Date of the collision. | Date in DD/MM/YYYY format. |
| `day_of_week` | Day of the week on which the collision occurred. | Categorical code: 1 = Sunday, 2 = Monday, 3 = Tuesday; see guide for the full mapping. |
| `time` | Recorded time of the collision. | Time in HH:MM format; null when unknown. |
| `local_authority_district` | District local-authority classification associated with the collision. | Categorical district code; for example, 1 = Westminster. Distinct from an ONS code. |
| `local_authority_ons_district` | Office for National Statistics district code associated with the collision location. | Geographic identifier; candidate field for borough selection. Validate against a suitable boundary. |
| `local_authority_highway` | Highway-authority geographic code recorded for the collision. | Geographic identifier; do not assume it identifies the same area as the district field. |
| `local_authority_highway_current` | Highway-authority geography mapped to the latest ONS boundaries using the recorded collision location. | Geographic identifier; introduced with the 2024 release and mapped retrospectively back to 2004, according to the guide. |
| `first_road_class` | Classification of the first recorded road. | Categorical code; examples include Motorway, A(M) and A. |
| `first_road_number` | Number of the first recorded road. | Road identifier, not a measurement; -1 = Unknown. Zero may indicate an unnumbered road. |
| `road_type` | Recorded road layout or type. | Categorical code; examples include roundabout, one-way street and dual carriageway. |
| `speed_limit` | Recorded speed limit of the road. | Numeric field with special codes: -1 = missing/out of range; 99 = unknown in self-reported data. Confirm units in the reporting documentation before using. |
| `junction_detail_historic` | Junction type under the 2011 or earlier STATS19 specification. | Categorical code; historical version. Codes must not be mixed directly with the newer field. |
| `junction_detail` | Junction type collected in or converted to the 2024 STATS19 specification. | Categorical code; for example, 13 = T or staggered junction, 16 = Crossroads. |
| `junction_control` | Type of junction control. | Categorical code; examples include authorised person and automatic traffic signal. Zero denotes not at a junction or within 20 metres. |
| `second_road_class` | Classification of the second recorded road, where applicable. | Categorical code; zero denotes not at a junction or within 20 metres. |
| `second_road_number` | Number of the second recorded road, where applicable. | Road identifier with special codes. Consult the guide for zero/unknown interpretation; do not treat as a continuous quantity. |
| `pedestrian_crossing_human_control_historic` | Human control of a nearby pedestrian crossing under the historical specification. | Categorical code; examples include school crossing patrol and other authorised person. |
| `pedestrian_crossing_physical_facilities_historic` | Nearby physical pedestrian-crossing facilities under the historical specification. | Categorical code; examples include zebra and signal-controlled crossings. Zero indicates none within 50 metres. |
| `pedestrian_crossing` | Pedestrian-crossing provision collected in or converted to the 2024 specification. | Categorical code covering crossing provision, including human control. Use its own mapping rather than historical codes. |
| `light_conditions` | Recorded lighting conditions at the collision. | Categorical code; for example, 1 = Daylight, 4 = Darkness with lights lit. |
| `weather_conditions` | Recorded weather conditions. | Categorical code; examples include fine weather, rain and snow, with wind categories. |
| `road_surface_conditions` | Recorded condition of the road surface. | Categorical code; for example, 1 = Dry, 2 = Wet or damp, 3 = Snow. |
| `special_conditions_at_site` | Recorded special conditions affecting the site. | Categorical code; examples include traffic-signal faults. Zero = None. |
| `carriageway_hazards_historic` | Recorded carriageway hazards under the historical specification. | Categorical code; examples include a vehicle load or another object on the road. |
| `carriageway_hazards` | Carriageway hazards collected in or converted to the 2024 specification. | Categorical code; use the newer mapping separately from the historical field. |
| `urban_or_rural_area` | Urban/rural classification of the location. | Categorical code: 1 = Urban, 2 = Rural, 3 = Unallocated. |
| `did_police_officer_attend_scene_of_accident` | Whether a police officer attended the collision scene and, for one category, whether it was self-reported. | Categorical code: 1 = Yes, 2 = No, 3 = No, reported using a self-completion form. |
| `trunk_road_flag` | Whether the collision location was recorded as a trunk road. | Categorical code: 1 = Trunk, 2 = Non-trunk, -1 = Missing/out of range. |
| `lsoa_of_accident_location` | Lower-layer Super Output Area (LSOA) code for the collision location. | Geographic identifier; England and Wales only. Boundary vintage requires confirmation before joining other sources. |
| `enhanced_severity_collision` | Enhanced collision-severity field. | The supplied code-list sheet does not describe this field. Confirm its precise definition and codes in DfT guidance before use. |
| `collision_injury_based` | Whether collision severity uses severity reporting or injury-code reporting. | Categorical flag: 0 = Severity reporting, 1 = Injury-code reporting. |
| `collision_adjusted_severity_serious` | Collision-level severity-adjustment field labelled for the serious category. | Not defined in the supplied code-list sheet. Confirm interpretation, units and usage in the DfT severity-adjustment guidance before use. |
| `collision_adjusted_severity_slight` | Collision-level severity-adjustment field labelled for the slight category. | Not defined in the supplied code-list sheet. Confirm interpretation, units and usage in the DfT severity-adjustment guidance before use. |


**Casualty table — 24 columns**

| Column | Description | Data kind and interpretation |
| --- | --- | --- |
| `collision_index` | Identifier of the collision to which this casualty belongs. | Join key to the collision table; preserve as text. |
| `collision_year` | Year assigned to the associated collision. | Year; check consistency with the collision record. |
| `collision_ref_no` | Police reference for the associated collision within a year. | Identifier; not unique across years. |
| `vehicle_reference` | Reference to the associated vehicle within the collision. | Identifier; vehicle references are not globally unique. Vehicle files are not currently supplied. |
| `casualty_reference` | Reference identifying a casualty within a collision. | Identifier; the guide notes that historical years may identify casualties within a vehicle instead. Validate the composite key. |
| `casualty_class` | Broad role of the casualty. | Categorical code: 1 = Driver or rider, 2 = Passenger, 3 = Pedestrian. Main pedestrian-selection field. |
| `sex_of_casualty` | Recorded sex of the casualty. | Categorical code; 1 = Male, 2 = Female, 9 = Unknown in self-reported data. Consult the guide for other missing codes. |
| `age_of_casualty` | Recorded age of the casualty. | Age field; -1 = Missing/out of range. Do not include missing codes in age calculations. |
| `age_band_of_casualty` | Grouped age category of the casualty. | Ordinal category; for example, 1 = 0–5, 2 = 6–10, 3 = 11–15. Code values are not ages. |
| `casualty_severity` | Recorded severity of the individual casualty. | Ordinal category: 1 = Fatal, 2 = Serious, 3 = Slight. Include these categories for the proposed pedestrian collision selection. |
| `pedestrian_location` | Recorded position of a pedestrian in relation to the road or crossing. | Categorical code; 0 = Not a pedestrian. Other codes describe crossing and road positions. |
| `pedestrian_movement` | Recorded pedestrian movement in relation to traffic. | Categorical code; 0 = Not a pedestrian. Other values describe direction or movement circumstances. |
| `car_passenger` | Whether the casualty was a car passenger and their seating position. | Categorical code: 0 = Not a car passenger, 1 = Front seat, 2 = Rear seat. |
| `bus_or_coach_passenger` | Passenger activity or position for a bus or coach casualty. | Categorical code; 0 = Not a bus/coach passenger, 1 = Boarding, 2 = Alighting; see guide for other values. |
| `pedestrian_road_maintenance_worker` | Whether a pedestrian casualty was recorded as a road-maintenance worker. | Categorical code: 0 = No/not applicable, 1 = Yes, 2 = Not known. The guide marks a specification applicability note; verify before comparisons. |
| `casualty_type` | Detailed road-user type of the casualty. | Categorical code; 0 = Pedestrian, 1 = Cyclist, with other vehicle/user categories. Cross-check against casualty_class; coding changes exist. |
| `casualty_imd_decile` | Index of Multiple Deprivation (IMD) decile associated with the casualty. | Ordinal area-deprivation category; 1 = Most deprived 10%. Confirm geographic basis and index version before interpretation; not an individual deprivation measure. |
| `lsoa_of_casualty` | LSOA code associated with the casualty. | Geographic identifier for England and Wales; distinct from collision-location LSOA. Confirm the geographic basis before use. |
| `enhanced_casualty_severity` | More detailed recorded casualty-severity classification where collected. | Codes include 1 = Fatal, 5 = Very Serious, 6 = Moderately Serious, 7 = Less Serious, 3 = Slight; -1 = Missing/out of range. Introduced where collected in 2023. |
| `casualty_injury_based` | Whether casualty severity uses severity reporting or injury-code reporting. | Categorical flag: 0 = Severity reporting, 1 = Injury-code reporting. |
| `casualty_adjusted_severity_serious` | Casualty-level severity-adjustment field labelled for the serious category. | Not defined in the supplied code-list sheet. Confirm interpretation, units and usage in DfT severity-adjustment guidance before use. |
| `casualty_adjusted_severity_slight` | Casualty-level severity-adjustment field labelled for the slight category. | Not defined in the supplied code-list sheet. Confirm interpretation, units and usage in DfT severity-adjustment guidance before use. |
| `casualty_distance_banding` | Band describing distance between the collision and the casualty home postcode. | Ordinal distance category; examples include within 5 km and 5.001–10 km. Guide states introduced in 2023. |
| `escooter_flag` | Whether the casualty was using an e-scooter. | Categorical flag: 1 = Yes, 0 = No. Guide states introduced in 2025; column presence in earlier files does not establish historical availability. |

**Planned use:** the initial concentration study principally needs collision identifiers, dates, borough geography, coordinates, pedestrian class/type and injury severity. Other fields provide optional context; their presence does not require including them in the analysis. Each qualifying collision will be counted once, regardless of the number of pedestrian casualties. No analysis beyond this dictionary is included in this notebook at this stage.